# 10.6 Graph Data in Python

**Prerequisites:** 10.1 Introduction to SQL, 10.3 SQLite in Python, 10.5 Beyond Relational  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- Nodes, edges and properties - the graph data model
- Modelling a service-dependency graph as an ordinary relational table
- 🔴 Why a fixed-depth question needs one `JOIN` **per hop**
- **Recursive CTEs** - `WITH RECURSIVE`, in plain SQLite
- 🔴 Cycles, and the two ways to stop a recursive query running forever
- `networkx` for in-memory graph algorithms: reachability, shortest path, cycles
- **Cypher** and Neo4j - the same questions in a language built for them
- Where a graph database genuinely wins, and the honest case for not using one

---

## The data model

A graph is two things:

- **Nodes** — the things. A service, a person, an account.
- **Edges** — the relationships. `DEPENDS_ON`, `FOLLOWS`, `TRANSFERRED_TO`.

Both can carry **properties**, and edges have a **direction** and a **type**.

```
     (web) ──DEPENDS_ON──> (api) ──DEPENDS_ON──> (auth)
       │                     │                      │
       │                     └──DEPENDS_ON──> (cache)│
       └──DEPENDS_ON──> (cdn)                   │    │
                                                v    v
                                             (userdb)
```

### What makes it a *graph* problem

Not the fact that things are related — relational databases handle that fine. It is when the interesting questions are about **paths of unknown length**:

- *Everything* `web` depends on, however many hops away
- The **shortest** route between two nodes
- Which services are affected if `userdb` goes down — the blast radius

> **The distinction that matters.** "Which services does `api` directly depend on?" is a relational question. "Which services does `api` depend on *at any depth*?" is a graph question. The first needs one join. The second needs recursion.

## Our worked example

A small service dependency graph — 8 services, 9 edges. Small enough to check by eye, deep enough that the third hop is where SQL starts to hurt.

Notice that `userdb` is reachable from `web` by **three different routes**. That will matter shortly.

In [ ]:
# (upstream, downstream) - 'upstream DEPENDS_ON downstream'
EDGES = [
    ("web", "api"),
    ("web", "cdn"),
    ("api", "auth"),
    ("api", "cache"),
    ("api", "search"),
    ("auth", "userdb"),
    ("cache", "userdb"),
    ("search", "index"),
    ("index", "userdb"),
]

SERVICES = sorted({n for edge in EDGES for n in edge})
print(f"{len(SERVICES)} services:", ", ".join(SERVICES))
print(f"{len(EDGES)} dependencies")

---

# Part 1: the relational model

A graph in a relational database is just an **edge table** — two foreign keys per row. There is nothing clever about it, and for direct-neighbour questions it is perfect.

```
    CREATE TABLE dep (upstream TEXT, downstream TEXT)
                      ^^^^^^^^       ^^^^^^^^^^
                      one row per EDGE, not per node
```

In [ ]:
import sqlite3

db = sqlite3.connect(":memory:")
db.execute("CREATE TABLE dep (upstream TEXT NOT NULL, downstream TEXT NOT NULL)")
db.executemany("INSERT INTO dep VALUES (?, ?)", EDGES)

print("direct dependencies of 'api' - one join, trivial SQL:")
for (svc,) in db.execute("SELECT downstream FROM dep WHERE upstream=? ORDER BY downstream",
                        ("api",)):
    print("   ", svc)

### 🔴 Now ask for two hops. Then three.

Each additional hop needs **another self-join**. The query does not just get longer — its shape changes with the depth you happen to want, which means the *depth is baked into the SQL*.

```
  1 hop : FROM dep d1
  2 hops: FROM dep d1 JOIN dep d2 ON d2.upstream = d1.downstream
  3 hops: FROM dep d1 JOIN dep d2 ON ... JOIN dep d3 ON ...
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                      and you cannot write this at all if the depth is unknown
```

Watch the 3-hop result: **`userdb` appears twice**, because two different paths reach it. Each row is a *path*, not a *node*, so deduplicating is now your job too.

In [ ]:
print("1 hop from 'web':")
print("  ", db.execute(
    "SELECT d1.downstream FROM dep d1 WHERE d1.upstream='web'").fetchall())

print("\n2 hops from 'web':")
print("  ", db.execute("""
    SELECT d2.downstream
    FROM dep d1
    JOIN dep d2 ON d2.upstream = d1.downstream
    WHERE d1.upstream = 'web'
""").fetchall())

print("\n3 hops from 'web':")
print("  ", db.execute("""
    SELECT d3.downstream
    FROM dep d1
    JOIN dep d2 ON d2.upstream = d1.downstream
    JOIN dep d3 ON d3.upstream = d2.downstream
    WHERE d1.upstream = 'web'
""").fetchall())
print("   ^ 'userdb' twice - two distinct paths reach it. Rows are paths, not nodes.")

## `WITH RECURSIVE` - SQL's answer

SQL does have an answer, and it is worth knowing well: a **recursive common table expression**. It is standard SQL and SQLite, PostgreSQL, MySQL 8 and SQL Server all support it.

```
  WITH RECURSIVE reach(node, depth) AS (

      SELECT 'web', 0                    <-- the ANCHOR: where to start
      UNION ALL
      SELECT d.downstream, r.depth + 1   <-- the RECURSIVE step: applied over and
      FROM dep d JOIN reach r            <-- over to rows already in `reach`,
        ON d.upstream = r.node           <-- until it produces nothing new
      WHERE r.depth < 10                 <-- 🔴 the guard. See the next section.
  )
  SELECT ... FROM reach
```

Read it as: *start here; repeatedly follow one more edge from whatever you have already reached; stop when nothing new appears.*

`min(depth)` in the outer query collapses the multiple paths back down to one row per node — the shortest way to reach it.

In [ ]:
rows = db.execute("""
    WITH RECURSIVE reach(node, depth) AS (
        SELECT 'web', 0
        UNION ALL
        SELECT d.downstream, r.depth + 1
        FROM dep d
        JOIN reach r ON d.upstream = r.node
        WHERE r.depth < 10
    )
    SELECT node, min(depth) AS hops
    FROM reach
    GROUP BY node
    ORDER BY hops, node
""").fetchall()

print("everything 'web' depends on, at any depth:")
for node, hops in rows:
    print(f"   {hops} hop(s): {node}")

print("\nOne query, any depth. This is the right tool far more often than")
print("people assume - you do not need a graph database to do this.")

### 🔴 Cycles: how a recursive query runs forever

Everything above assumed the graph has no loops. Real dependency graphs grow them by accident, and a recursive CTE with no guard will happily follow a cycle **indefinitely** — the query never returns.

Two ways to stop it:

| Guard | How | Trade-off |
|---|---|---|
| **Depth limit** | `WHERE depth < 10` | Simple; silently truncates a genuinely deeper graph |
| **Path tracking** | carry the path, skip nodes already in it | Correct at any depth; slightly more SQL |

The path-tracking idiom is worth memorising:

```
  r.path || d.downstream || ','                    build a ',a,b,c,' trail
  WHERE instr(r.path, ',' || d.downstream || ',') = 0
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        "have I already been here on THIS path?" - the commas stop
        'api' from matching inside 'api-gateway'
```

In [ ]:
# Introduce a cycle: userdb -> web, closing the loop
db.execute("INSERT INTO dep VALUES ('userdb', 'web')")
print("added edge userdb -> web, so the graph now has a cycle\n")

# ---- Guard 1: a depth limit ----
capped = db.execute("""
    WITH RECURSIVE reach(node, depth) AS (
        SELECT 'web', 0
        UNION ALL
        SELECT d.downstream, r.depth + 1
        FROM dep d JOIN reach r ON d.upstream = r.node
        WHERE r.depth < 6
    )
    SELECT count(*) FROM reach
""").fetchone()[0]
print(f"depth-limited: {capped} rows produced before the cap stopped it")
print("  ^ without `WHERE r.depth < 6` this query never returns.\n")

# ---- Guard 2: path tracking - correct, and needs no arbitrary limit ----
safe = db.execute("""
    WITH RECURSIVE reach(node, depth, path) AS (
        SELECT 'web', 0, ',web,'
        UNION ALL
        SELECT d.downstream, r.depth + 1, r.path || d.downstream || ','
        FROM dep d JOIN reach r ON d.upstream = r.node
        WHERE instr(r.path, ',' || d.downstream || ',') = 0
    )
    SELECT node, min(depth) FROM reach GROUP BY node ORDER BY min(depth), node
""").fetchall()
print("path-tracked (cycle-safe, no artificial depth limit):")
for node, hops in safe:
    print(f"   {hops} hop(s): {node}")

db.execute("DELETE FROM dep WHERE upstream='userdb' AND downstream='web'")
print("\ncycle removed again")

---

# Part 2: `networkx` - graph algorithms in memory

Often the real answer. If the graph **fits in memory**, `networkx` gives you decades of graph algorithms for the cost of `pip install networkx` — no server, no new query language, no operational burden.

The catch is the same as any in-memory structure: it is not shared, not persistent, and bounded by RAM. Load it from your real database, compute, throw it away.

| Type | Meaning |
|---|---|
| `Graph` | undirected |
| `DiGraph` | directed — what a dependency graph needs |
| `MultiDiGraph` | directed, multiple edges between the same pair |

In [ ]:
import networkx as nx

g = nx.DiGraph()
g.add_edges_from(EDGES)
print(f"{g.number_of_nodes()} nodes, {g.number_of_edges()} edges\n")

# ---- The same question as the recursive CTE, in one call ----
print("descendants('web')  -- everything web depends on, any depth:")
print("   ", sorted(nx.descendants(g, "web")))

# ---- And the reverse: the blast radius ----
print("\nancestors('userdb') -- everything that breaks if userdb dies:")
print("   ", sorted(nx.ancestors(g, "userdb")))

print("\nmost depended-upon services (in-degree):")
for node, deg in sorted(g.in_degree(), key=lambda kv: -kv[1])[:3]:
    print(f"    {node:<8} {deg} direct dependents")

### Paths, ordering and cycles

> **Shortest paths are not unique.** `web` reaches `userdb` by three routes, two of which are the same length. `shortest_path` returns *a* shortest path, not *the* one — so `networkx` and Neo4j may legitimately give different answers of equal length. Do not assert on the specific path in a test; assert on its length.

In [ ]:
print("a shortest path web -> userdb:")
print("   ", nx.shortest_path(g, "web", "userdb"))

print("\nALL simple paths web -> userdb:")
for path in nx.all_simple_paths(g, "web", "userdb"):
    print("    " + " -> ".join(path))
print("   ^ this is why the 3-hop SQL returned 'userdb' more than once")

# ---- Is it safe to deploy in order? ----
print("\nis this a DAG (no circular dependencies)?", nx.is_directed_acyclic_graph(g))
print("a safe deploy order (topological sort):")
print("   ", " -> ".join(nx.topological_sort(g)))

# ---- Add a cycle and detect it ----
cyclic = g.copy()
cyclic.add_edge("userdb", "web")
print("\nafter adding userdb -> web:")
print("    is a DAG:", nx.is_directed_acyclic_graph(cyclic))
cycle_edges = nx.find_cycle(cyclic)
# find_cycle returns edges; chain the sources then close the loop back
# to the first node, so the output actually reads as a cycle.
loop = [a for a, _ in cycle_edges] + [cycle_edges[0][0]]
print("    cycle   :", " -> ".join(loop))
print("   ^ exactly the check you want in CI to stop circular dependencies")

---

# Part 3: Neo4j and Cypher

A graph **database** stores nodes and edges as its native structures, so following an edge is a pointer hop rather than an index lookup and a join. Its query language, **Cypher**, draws the pattern you are looking for:

```
    MATCH (a:Svc {name:'web'})-[:DEPENDS_ON*]->(x)
          ^^^^^^^^^^^^^^^^^^^^  ^^^^^^^^^^^^^^^  ^^^
          a node, labelled Svc  an edge type,    any node
                                * = ANY depth
```

The `*` is the whole argument for graph databases. It is one character where SQL needs a recursive CTE, and the depth need never be decided in advance.

| Question | Cypher | SQL |
|---|---|---|
| direct deps | `(a)-[:DEPENDS_ON]->(x)` | one join |
| any depth | `(a)-[:DEPENDS_ON*]->(x)` | recursive CTE |
| 2 to 4 hops | `(a)-[:DEPENDS_ON*2..4]->(x)` | three unioned queries |
| shortest | `shortestPath((a)-[*]->(b))` | recursive CTE + `min()` + care |

This notebook runs without Neo4j. The next cell checks.

In [ ]:
import os
import socket


def server_available(host: str, port: int, timeout: float = 0.5) -> bool:
    with socket.socket() as probe:
        probe.settimeout(timeout)
        try:
            probe.connect((host, port))
            return True
        except OSError:
            return False


HOST = os.environ.get("PYNOTES_DB_HOST", "127.0.0.1")
NEO4J_PORT = int(os.environ.get("PYNOTES_NEO4J_PORT", 57687))
HAVE_NEO4J = server_available(HOST, NEO4J_PORT)

print(f"Neo4j bolt on {HOST}:{NEO4J_PORT} ->",
      "reachable" if HAVE_NEO4J else "not running")

if not HAVE_NEO4J:
    print(
        "\nThe Cypher below will be shown but not executed. Every question it\n"
        "answers was already answered above with sqlite3 and networkx.\n"
        'To run it for real:  docker compose -f "10 Database/docker/docker-compose.yml" up -d'
    )

In [ ]:
driver = None

CYPHER_LOAD = """
UNWIND $edges AS e
MERGE (a:Svc {name: e[0]})
MERGE (b:Svc {name: e[1]})
MERGE (a)-[:DEPENDS_ON]->(b)
"""

if HAVE_NEO4J:
    from neo4j import GraphDatabase

    driver = GraphDatabase.driver(
        f"bolt://{HOST}:{NEO4J_PORT}",
        auth=(
            os.environ.get("PYNOTES_NEO4J_USER", "neo4j"),
            os.environ.get("PYNOTES_NEO4J_PASSWORD", "learnpython"),
        ),
    )
    driver.verify_connectivity()

    with driver.session() as session:
        session.run("MATCH (n:Svc) DETACH DELETE n")        # re-runnable
        # MERGE is idempotent: create if absent, match if present.
        session.run(CYPHER_LOAD, edges=[list(e) for e in EDGES])
        n = session.run("MATCH (n:Svc) RETURN count(n) AS n").single()["n"]
        r = session.run(
            "MATCH ()-[r:DEPENDS_ON]->() RETURN count(r) AS r").single()["r"]
    print(f"loaded {n} nodes and {r} relationships")
else:
    print("Would have run:")
    print(CYPHER_LOAD)

In [ ]:
QUERIES = [
    (
        "everything 'web' depends on, ANY depth",
        "MATCH (:Svc {name:'web'})-[:DEPENDS_ON*]->(x:Svc) "
        "RETURN DISTINCT x.name AS service ORDER BY service",
    ),
    (
        "blast radius: what breaks if 'userdb' dies",
        "MATCH (x:Svc)-[:DEPENDS_ON*]->(:Svc {name:'userdb'}) "
        "RETURN DISTINCT x.name AS service ORDER BY service",
    ),
    (
        "exactly 2 hops away",
        "MATCH (:Svc {name:'web'})-[:DEPENDS_ON*2]->(x:Svc) "
        "RETURN DISTINCT x.name AS service ORDER BY service",
    ),
]

for description, cypher in QUERIES:
    print(f"-- {description} --")
    print(f"   {cypher}")
    if driver is not None:
        with driver.session() as session:
            got = [rec["service"] for rec in session.run(cypher)]
        print("   ->", got)
    else:
        print("   -> [not executed: no server]")
    print()

In [ ]:
SHORTEST = (
    "MATCH p = shortestPath( "
    "  (:Svc {name:'web'})-[:DEPENDS_ON*]->(:Svc {name:'userdb'}) ) "
    "RETURN [n IN nodes(p) | n.name] AS path"
)

print("-- shortest path web -> userdb --")
print(f"   {SHORTEST}\n")

if driver is not None:
    with driver.session() as session:
        cypher_path = session.run(SHORTEST).single()["path"]
    print("   Cypher   ->", " -> ".join(cypher_path))
else:
    cypher_path = None
    print("   Cypher   -> [not executed: no server]")

nx_path = nx.shortest_path(g, "web", "userdb")
print("   networkx ->", " -> ".join(nx_path))

if cypher_path is not None and cypher_path != nx_path:
    print("\n   The two answers differ - and both are correct.")
    print(f"   Both are {len(nx_path) - 1} hops; there is more than one shortest route.")
    print("   Assert on the LENGTH of a shortest path, never on its identity.")

---

## The same question, three ways

*"Everything `web` depends on, at any depth."*

| | Code | Needs |
|---|---|---|
| **SQL** | a 9-line `WITH RECURSIVE`, plus a cycle guard | nothing you do not already have |
| **networkx** | `nx.descendants(g, 'web')` | the graph to fit in memory |
| **Cypher** | `MATCH (:Svc{name:'web'})-[:DEPENDS_ON*]->(x)` | a database to run and operate |

All three returned the same seven services. That is the honest headline: **for a graph this size, all three are fine**, and the recursive CTE needs no new infrastructure at all.

## So when is a graph database actually worth it?

The cost is real: another database to run, back up, secure and monitor, and a query language your team does not know. Pay it when **more than one** of these is true:

- Traversals are **deep** — routinely more than 3–4 hops. This is where a join per hop stops being viable and index lookups pile up.
- The graph is **large and dense**, and joins are producing intermediate result sets far bigger than the answer.
- Path queries are your **main workload**, not an occasional report.
- You need path-aware algorithms — shortest path, centrality, community detection — **as queries**, continuously, over data too big to hold in memory.

### And when it is not

| Situation | Do this instead |
|---|---|
| A few thousand nodes | Load it into `networkx` |
| Depth 1–3 | Ordinary joins |
| Unknown depth, moderate size | `WITH RECURSIVE` |
| One report per week | `WITH RECURSIVE` |
| Mostly aggregation, occasional traversal | Relational, or columnar (**10.5**) |

> **The common mistake is adopting a graph database because the data *is* a graph.** Almost all data is a graph if you squint. What justifies the move is the **query pattern**, not the data model — and you should be able to point at a specific query that is too slow or impossible to express today.

In [ ]:
# ---- tidy up ----
if driver is not None:
    with driver.session() as session:
        session.run("MATCH (n:Svc) DETACH DELETE n")
    driver.close()
    print("neo4j : nodes deleted, driver closed")

db.close()
print("sqlite: connection closed")
print("\nnetworkx held everything in memory - nothing to clean up.")

---

## Common Mistakes & Pitfalls

1. 🔴 **A recursive CTE with no guard on a cyclic graph.** It never returns. Use a depth limit or track the path.
2. 🔴 **Forgetting that rows are paths, not nodes.** A multi-hop join returns a node once per route that reaches it. `DISTINCT` or `GROUP BY` if you want nodes.
3. **Asserting on a specific shortest path.** More than one may be equally short; different tools legitimately return different ones. Assert on the length.
4. **Substring matching when tracking paths.** Without the comma delimiters, `'api'` matches inside `'api-gateway'` and silently prunes a valid branch.
5. **Adopting a graph database because the data is a graph.** The query pattern is what justifies it, not the shape of the data.
6. **Reaching for `networkx` on a graph that does not fit in memory.** It is an in-memory library; there is no spill-to-disk.
7. **Assuming direction does not matter.** `DEPENDS_ON` reversed answers a completely different question — dependencies vs blast radius.
8. **Using `CREATE` instead of `MERGE` in Cypher when reloading.** `CREATE` duplicates nodes on every run; `MERGE` is idempotent.

## Best Practices

- Try `WITH RECURSIVE` before adding a graph database. It is standard SQL and it is usually enough.
- For a graph that fits in memory, load it into `networkx`, compute, discard.
- Always guard recursive queries against cycles, even when you are sure there are none.
- Store edges in their natural direction and traverse backwards when you need the reverse; do not duplicate rows.
- Run a cycle check (`nx.is_directed_acyclic_graph`) in CI for anything that must stay acyclic — dependencies, build order, schedules.
- Index the columns you traverse: `dep(upstream)` and `dep(downstream)`.
- Use `MERGE` for idempotent loads in Cypher, and label every node.
- Close the Neo4j driver explicitly; it holds a connection pool.

## Practice Exercises

Try these before moving on.

1. Add an index on `dep(upstream)` and compare `EXPLAIN QUERY PLAN` for the recursive CTE before and after.
2. Rewrite the recursive CTE to answer the reverse question — everything that depends on `userdb` — and check it against `nx.ancestors`.
3. Give each edge a `latency_ms` property and find the *cheapest* path rather than the shortest, with `nx.shortest_path(..., weight='latency_ms')`.
4. Write a function that returns the deployment order for a set of services, raising a clear error naming the cycle if one exists.
5. 🔴 Remove the `WHERE r.depth < 6` guard from the depth-limited query, re-add the `userdb -> web` edge, and predict what happens **before** running it. Then run it with a small `LIMIT` to confirm safely.
6. Load 10,000 random edges into both SQLite and Neo4j and compare a 5-hop traversal. At what depth does the difference start to matter?
7. Model something other than services — a social graph, or a folder tree — and decide which of the three approaches fits it best. Justify the choice.